In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 1. Define transformations for data preprocessing
#    - Convert images to PyTorch tensors using ToTensor()
#    - Normalize pixel values to the range -1 ~ 1 using Normalize()
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# 2. Download and load the training dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

# 3. Download and load the test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

# 4. Define the names of the 10 classes
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 5. Function to visualize and check some images from the dataset
def imshow(img):
    img = img / 2 + 0.5     # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0))) # (Channel, Height, Width) -> (Height, Width, Channel)
    plt.show()

# Get a batch of images from the training data
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Print images and labels
imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Define our own CNN model class by inheriting from nn.Module
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1. Define convolutional/pooling layers
        # Input channel 3, output channel 6, 5x5 kernel
        self.conv1 = nn.Conv2d(3, 6, 5)
        # 2x2 max pooling
        self.pool = nn.MaxPool2d(2, 2)
        # Input channel 6, output channel 16, 5x5 kernel
        self.conv2 = nn.Conv2d(6, 16, 5)

        # 2. Define fully connected layers
        # Input node 16*5*5, output node 120
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        # The final output is the score for 10 classes
        self.fc3 = nn.Linear(84, 10)

    # Define the flow of data through the model (forward propagation)
    def forward(self, x):
        # Convolution -> Activation (ReLU) -> Pooling
        x = self.pool(F.relu(self.conv1(x)))
        # Convolution -> Activation (ReLU) -> Pooling
        x = self.pool(F.relu(self.conv2(x)))
        
        # Flatten the 3D feature map into a 1D vector
        # -1 means automatically calculate the batch size
        x = x.view(-1, 16 * 5 * 5)
        
        # Fully connected layer -> Activation (ReLU)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Do not apply an activation function to the last output layer
        # (CrossEntropyLoss handles it internally)
        x = self.fc3(x)
        return x

# Create model object
net = Net()

# Move the model to the GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
net.to(device)

In [ ]:
import torch.optim as optim

# 1. Use cross-entropy loss as the loss function
criterion = nn.CrossEntropyLoss()

# 2. Use SGD as the optimizer.
#    - net.parameters(): Pass the parameters to be trained
#    - lr=0.001: Learning rate
#    - momentum=0.9: Momentum value
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
print("Starting Training...")

# Train for 2 epochs
for epoch in range(2):  
    running_loss = 0.0
    # Get data in mini-batches from trainloader
    for i, data in enumerate(trainloader, 0):
        # 1. Get data (inputs and labels) and move to GPU
        inputs, labels = data[0].to(device), data[1].to(device)

        # 2. Initialize gradients
        optimizer.zero_grad()

        # 3. Forward propagation -> 4. Calculate loss -> 5. Backpropagation -> 6. Update parameters
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Print loss value every 2000 mini-batches
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

# Save the parameters of the trained model
PATH = './cifar_net.pth'
torch.save(net.state_dict(), PATH)

In [ ]:
# Calculate the accuracy for the entire test dataset
correct = 0
total = 0

# Do not calculate gradients in evaluation mode
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        # Select the class with the highest score as the prediction result
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

# Check the accuracy for each class
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

for i in range(10):
    print(f'Accuracy of {classes[i]:5s} : {100 * class_correct[i] / class_total[i]:.1f} %')